# Rossmann: LightGBM với lag feature, không leakage

Rossmann (chuỗi nhà thuốc Đức) cần dự báo doanh thu **từng cửa hàng, từng ngày, cho 6 tuần tới** để
xếp lịch nhân viên. Đây là đề của cuộc thi Kaggle 2015, dữ liệu thật: 1 115 cửa hàng, 2013-01-01 tới
2015-07-31.

Bài này dùng 50 cửa hàng có id nhỏ nhất trong số cửa hàng đủ mọi ngày (180 cửa hàng thiếu nửa sau
năm 2014), để chạy nhanh. Model là **một** LightGBM chung cho mọi cửa hàng (global model), feature là
lag và rolling của chính doanh thu, cộng các biến biết trước (lịch khuyến mãi, ngày lễ).

Quy tắc vàng: dự báo cho ngày `t` lập ở ngày `t − 42` thì chỉ được dùng doanh thu tới `t − 42`.
Mọi feature từ `Sales` phải lùi ít nhất `horizon` ngày. `Customers` là con số chỉ biết sau khi ngày
đó kết thúc: không được dùng làm feature.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm hoặc hằng.
- Cell gắn tag `explore` (dòng đầu `# explore`) đọc dữ liệu thật, vẽ, gọi hàm. Test bỏ qua chúng.
- Chấm: `uv run pytest tracks/data-science/modules/07-time-series/exercises/intermediate/03-store-sales-lightgbm -q`

In [ ]:
import lightgbm as lgb
import numpy as np
import pandas as pd

HORIZON = 42
FEATURES = ["lag_h", "lag_h_7", "lag_h_14", "lag_364", "roll_mean_28", "dow_mean_8w",
            "DayOfWeek", "Promo", "SchoolHoliday", "StateHoliday", "month", "day"]
LGBM_PARAMS = dict(n_estimators=500, learning_rate=0.05, num_leaves=31, random_state=0, verbose=-1)

## 1. Chọn cửa hàng

`select_stores(train, n_stores=50)`: giữ `n_stores` cửa hàng có id nhỏ nhất trong số cửa hàng có đủ
**mọi** ngày xuất hiện trong dữ liệu; sắp theo `Store`, `Date`, `reset_index(drop=True)`. Đủ ngày
nghĩa là mỗi cửa hàng có đúng một dòng mỗi ngày, nên "lùi k dòng" trong một cửa hàng là "lùi k ngày".

In [ ]:
def select_stores(train: pd.DataFrame, n_stores: int = 50) -> pd.DataFrame:
    """n_stores cửa hàng id nhỏ nhất trong số cửa hàng đủ mọi ngày; sắp theo Store, Date."""
    n_days = train.Date.nunique()
    counts = train.groupby("Store").Date.nunique()
    keep = counts[counts == n_days].index.sort_values()[:n_stores]
    return train[train.Store.isin(keep)].sort_values(["Store", "Date"]).reset_index(drop=True)

In [ ]:
# explore: đọc dữ liệu thật (tải bằng: uv run python data/download_dataset.py store-sales)
from pathlib import Path

import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
raw = pd.read_csv(ROOT / "data/raw/store-sales/train.csv", dtype={"StateHoliday": str}, parse_dates=["Date"])
df = select_stores(raw)
print(raw.shape, df.shape, df.Store.nunique(), "cửa hàng")
s1 = df[df.Store == 1].set_index("Date")
s1.Sales["2015-03-01":].plot(figsize=(11, 3), title="Cửa hàng 1: Chủ nhật đóng cửa (0), tuần khuyến mãi cao hơn")
plt.show()
df[df.Open == 1].groupby(["DayOfWeek", "Promo"]).Sales.mean().unstack().round(0)

## 2. Feature không leakage

`make_features(df, horizon=HORIZON)` trả DataFrame cùng index với `df`, đúng các cột `FEATURES`
theo thứ tự. Đặt `sales_open = Sales` ở ngày mở cửa, `NaN` ở ngày đóng cửa (ngày đóng cửa không nói
gì về sức mua). Mọi phép lùi làm **trong từng cửa hàng**:

| Cột | Định nghĩa |
| --- | --- |
| `lag_h`, `lag_h_7`, `lag_h_14` | `sales_open` lùi `horizon`, `horizon + 7`, `horizon + 14` ngày |
| `lag_364` | `sales_open` lùi 364 ngày (cùng thứ, năm trước) |
| `roll_mean_28` | trung bình 28 giá trị của `lag_h` kết thúc ở dòng hiện tại (`min_periods=1`, bỏ NaN) |
| `dow_mean_8w` | trung bình `sales_open` lùi `d, d + 7, …, d + 49` ngày với `d = 7 * ceil(horizon / 7)` (8 tuần gần nhất cùng thứ, bỏ NaN) |
| `DayOfWeek`, `Promo` | giữ nguyên (lịch khuyến mãi biết trước) |
| `SchoolHoliday`, `StateHoliday` | số nguyên 0/1; `StateHoliday` là 1 khi khác `"0"` |
| `month`, `day` | từ `Date` |

In [ ]:
def make_features(df: pd.DataFrame, horizon: int = HORIZON) -> pd.DataFrame:
    """Feature cho mỗi dòng (Store, Date), chỉ dùng Sales tới Date - horizon. Cột = FEATURES."""
    df = df.sort_values(["Store", "Date"])
    sales_open = df.Sales.where(df.Open == 1)
    by_store = sales_open.groupby(df.Store)
    out = pd.DataFrame(index=df.index)
    out["lag_h"] = by_store.shift(horizon)
    out["lag_h_7"] = by_store.shift(horizon + 7)
    out["lag_h_14"] = by_store.shift(horizon + 14)
    out["lag_364"] = by_store.shift(364)
    out["roll_mean_28"] = (out["lag_h"].groupby(df.Store)
                           .transform(lambda x: x.rolling(28, min_periods=1).mean()))
    first = 7 * int(np.ceil(horizon / 7))
    same_dow = pd.concat([by_store.shift(first + 7 * k) for k in range(8)], axis=1)
    out["dow_mean_8w"] = same_dow.mean(axis=1)
    out["DayOfWeek"] = df.DayOfWeek
    out["Promo"] = df.Promo
    out["SchoolHoliday"] = df.SchoolHoliday.astype(int)
    out["StateHoliday"] = (df.StateHoliday.astype(str) != "0").astype(int)
    out["month"] = df.Date.dt.month
    out["day"] = df.Date.dt.day
    return out[FEATURES].sort_index()

## 3. Train, dự báo và baseline

`fit_predict(df, cutoff, horizon=HORIZON)`:
- Feature từ `make_features(df, horizon)` trên toàn bộ `df` (an toàn vì feature đã lùi đủ xa).
- Train: dòng có `Date < cutoff` và `Open == 1`; target `log1p(Sales)`; `lgb.LGBMRegressor(**LGBM_PARAMS)`.
- Test: dòng có `cutoff <= Date < cutoff + horizon`. Dự báo `expm1(...)`, ngày đóng cửa là 0.
- Trả `pd.Series` dự báo, index là index các dòng test của `df`.

`seasonal_naive_last_week(df, cutoff, horizon)`: mỗi cửa hàng, dự báo bằng doanh thu cùng thứ trong
7 ngày ngay trước cutoff; ngày đóng cửa là 0. Cùng kiểu trả về.

`evaluate(df, pred)`: trên các dòng của `pred` có `Open == 1` và `Sales > 0`, trả
`{"mae": ..., "rmspe": ...}` với RMSPE là metric của cuộc thi.

In [ ]:
def fit_predict(df: pd.DataFrame, cutoff, horizon: int = HORIZON) -> pd.Series:
    """Dự báo horizon ngày từ cutoff cho mọi cửa hàng trong df."""
    cutoff = pd.Timestamp(cutoff)
    X = make_features(df, horizon)
    train = (df.Date < cutoff) & (df.Open == 1)
    test = (df.Date >= cutoff) & (df.Date < cutoff + pd.Timedelta(days=horizon))
    model = lgb.LGBMRegressor(**LGBM_PARAMS)
    model.fit(X[train], np.log1p(df.Sales[train]))
    pred = np.expm1(model.predict(X[test]))
    return pd.Series(np.where(df.Open[test] == 1, pred, 0.0), index=df.index[test])


def seasonal_naive_last_week(df: pd.DataFrame, cutoff, horizon: int = HORIZON) -> pd.Series:
    """Doanh thu cùng thứ của tuần ngay trước cutoff; ngày đóng cửa là 0."""
    cutoff = pd.Timestamp(cutoff)
    last_week = df[(df.Date >= cutoff - pd.Timedelta(days=7)) & (df.Date < cutoff)]
    lookup = last_week.set_index(["Store", "DayOfWeek"]).Sales
    test = df[(df.Date >= cutoff) & (df.Date < cutoff + pd.Timedelta(days=horizon))]
    pred = pd.MultiIndex.from_frame(test[["Store", "DayOfWeek"]]).map(lookup).to_numpy(dtype=float)
    return pd.Series(np.where(test.Open == 1, pred, 0.0), index=test.index)


def evaluate(df: pd.DataFrame, pred: pd.Series) -> dict:
    """MAE và RMSPE trên dòng mở cửa có Sales > 0."""
    rows = df.loc[pred.index]
    mask = (rows.Open == 1) & (rows.Sales > 0)
    y, p = rows.Sales[mask].to_numpy(float), pred[mask].to_numpy(float)
    return {"mae": float(np.mean(np.abs(y - p))), "rmspe": float(np.sqrt(np.mean(((y - p) / y) ** 2)))}

In [ ]:
# explore: 6 tuần cuối (2015-06-20 tới 2015-07-31) làm test
CUTOFF = "2015-06-20"
pred = fit_predict(df, CUTOFF)
naive = seasonal_naive_last_week(df, CUTOFF)
print("LightGBM      ", evaluate(df, pred))
print("seasonal naive", evaluate(df, naive))

In [ ]:
# explore: backtest thêm hai origin trước đó: kết luận có giữ không?
for c in ["2015-03-28", "2015-05-09"]:
    print(c, evaluate(df, fit_predict(df, c)), evaluate(df, seasonal_naive_last_week(df, c)))

In [ ]:
# explore: leakage trông như thế nào. Thêm lag 1 và lag 7 (không thể có khi dự báo 42 ngày tới)
X = make_features(df)
by_store = df.Sales.where(df.Open == 1).groupby(df.Store)
X["lag_1"], X["lag_7"] = by_store.shift(1), by_store.shift(7)
c = pd.Timestamp(CUTOFF)
tr, te = (df.Date < c) & (df.Open == 1), df.Date >= c
leaky = lgb.LGBMRegressor(**LGBM_PARAMS).fit(X[tr], np.log1p(df.Sales[tr]))
leaky_pred = pd.Series(np.where(df.Open[te] == 1, np.expm1(leaky.predict(X[te])), 0.0), index=df.index[te])
print("có leakage", evaluate(df, leaky_pred), "<- đẹp hơn, nhưng là số giả")

In [ ]:
# explore: sai số theo tuần của horizon và feature importance
rows = df.loc[pred.index]
err = (rows.Sales - pred).abs()[rows.Open == 1]
week = ((rows.Date - pd.Timestamp(CUTOFF)).dt.days // 7)[rows.Open == 1]
print(err.groupby(week).mean().round(0))
model = lgb.LGBMRegressor(**LGBM_PARAMS).fit(X.loc[tr, FEATURES], np.log1p(df.Sales[tr]))
pd.Series(model.feature_importances_, FEATURES).sort_values().plot.barh(title="split importance")
plt.show()

## 4. Nhận xét

Ở origin 2015-06-20, LightGBM có MAE khoảng 665 và RMSPE 0.138, so với 1 501 và 0.350 của seasonal
naive: sai số giảm hơn một nửa. Baseline "tuần trước" thua nặng vì Rossmann chạy khuyến mãi xen kẽ
theo tuần, nên tuần trước thường có trạng thái `Promo` ngược với tuần cần dự báo; model biết lịch
`Promo` trước nên xử lý được. Kết luận giữ ở hai origin trước đó (2015-03-28, 2015-05-09). Bản có
`lag_1`, `lag_7` cho MAE khoảng 575: đẹp hơn, nhưng lúc dự báo ngày thứ 30 ta không có doanh thu của
ngày thứ 29, nên con số đó không bao giờ đạt được trong production. Với horizon 7 ngày, có thể dùng
lag 7, 14 và rolling kết thúc ở `t − 7`; hoặc huấn luyện một model riêng cho từng khoảng horizon
(direct strategy).